In [10]:
from gpt2_model import GPT2, DataLoader, ShardedDataLoader, GPTConfig
from train_gpt2 import GPTTrainer
from prepare_data import DATASET_REGISTRY
from torchinfo import summary
import torch
import os
from torch.nn.parallel import DistributedDataParallel as DDP

In [11]:
# Set precision level
# torch.set_float32_matmul_precision('high')

# check support for bfloats

x = torch.tensor([1.0], dtype=torch.bfloat16).to('mps')
x.dtype

torch.bfloat16

In [ ]:
# MAX_ITRS = 10
# config = GPTConfig(
#     context_len=1024,
#     batch_size=8,
#     emb_dim=768,
#     n_heads=12,
#     n_transformer_blocks=12,
#     dropout=0.05,
#     vocab_size=50304, # padded to nearest power of 2 for efficiency
#     use_flash_attn=False,
#     warmup_iters=10,
#     learning_rate=6e-4,
#     max_train_iters=MAX_ITRS,
#     min_lr=6e-5,       # 10% of max LR
#     weight_decay=0.1,
#     total_batch_size=524288, # 2^19
# )
# config.grad_acum_steps

64

In [12]:
MAX_ITRS = 20

config = GPTConfig(
    context_len=1024,
    batch_size=8,
    emb_dim=768,
    n_heads=12,
    n_transformer_blocks=12,
    dropout=0.05,
    vocab_size=50304, # padded to nearest power of 2 for efficiency
    use_flash_attn=False,
    warmup_iters=10,
    learning_rate=6e-4,
    max_train_iters=MAX_ITRS,
    min_lr=6e-5,        # 10% of max LR
    weight_decay=0.1,
    total_batch_size=524288, # 2^19 — for multi-GPU CUDA; override below for single-device runs
)
assert config.total_batch_size % (config.batch_size * config.context_len) == 0, "Make sure total batch size is divisiible by (B*CL)"

dpp_info = GPTTrainer.setup_ddp(config=config)

# world_size number of processes will be training simultaneously, so we can scale
# down the desired gradient accumulation iterations per process proportionally
assert config.total_batch_size % (config.batch_size * config.context_len * dpp_info.world_size) == 0, "Make sure total batch size is divisiible by (B*CL* world_size)"

# We are dividing(reducing) the no of micro steps to take, since in DDP we will spin up `ddp_world_size` no. of processes
config.grad_acum_steps = config.total_batch_size // (config.batch_size * config.context_len * dpp_info.world_size)
config.grad_acum_steps

tokens per iteration will be: 524,288


64

In [13]:
device = dpp_info.device
if dpp_info.master_process:
    print(f"Total batch size = {config.total_batch_size}")
    print(f"grad_acum_steps = {config.grad_acum_steps}")
    print(f"Using device: {device}")

Total batch size = 524288
grad_acum_steps = 64
Using device: mps


In [14]:

model = GPT2(config)

In [15]:
# device = "cpu"
# if torch.cuda.is_available():
#     device = "cuda"
# elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
#     device = "mps"

model.to(device)
# model = torch.compile(model)

if dpp_info.ddp:
        model = DDP(model, device_ids=[dpp_info.local_rank])

In [7]:
PROJECT_ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
BOOKS_DIR    = f"{PROJECT_ROOT}/main/resources/books"
book_files = [
    # "complete_mb.txt",
    # "val_ramn.txt",
    "tinyshakespeare.txt",
]

for fname in book_files:
    path = f"{BOOKS_DIR}/{fname}"
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()

print(f"Total len of text = {len(text)}")
print(text[:50])

Total len of text = 1115394
First Citizen:
Before we proceed any further, hear


In [18]:
data_loader = DataLoader(text, batch_size=config.batch_size, context_len=config.context_len, num_processes=dpp_info.world_size, process_rank=dpp_info.rank)

Loaded 338025 tokens
One complete epoch = 41 batches


In [16]:
# ── ShardedDataLoader ────────────────────────────────────────────
# Use this instead of the DataLoader cell above when training on pre-tokenized
# shards from prepare_data.py (FineWeb 55% + Books 25% + TinyStories 20%).
#
# Run prepare_data.py first:
#   python prepare_data.py --max_tokens 10_000_000    # quick test
#   python prepare_data.py                             # full datasets
#
# Then uncomment below and comment out the DataLoader cell above.


PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
data_dir = os.path.join(PROJECT_ROOT, "data")
dataset_weights = {name: cfg["weight"] for name, cfg in DATASET_REGISTRY.items()}
loader_args = dict(data_dir=data_dir, dataset_weights=dataset_weights,
                    batch_size=config.batch_size, context_len=config.context_len,
                    num_processes=dpp_info.world_size, process_rank=dpp_info.rank)
data_loader = ShardedDataLoader(**loader_args, split="train")
val_loader  = ShardedDataLoader(**loader_args, split="val")
test_loader = ShardedDataLoader(**loader_args, split="test")

ShardedDataLoader [train]: 'fineweb' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [train]: 'books' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [train]: 'tinystories' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [val]: 'fineweb' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [val]: 'books' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [val]: 'tinystories' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'fineweb' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'books' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'tinystories' — 1 shard(s), 5,000,000 tokens


In [17]:
config.max_train_iters=1
trainer = GPTTrainer(model, config, data_loader, device, val_loader=val_loader, test_loader=test_loader)

Number of decay params = 50 params with sum 124354560, no. of non decay params = 98 params with sum 121344


In [18]:
total_tox = data_loader.total_tokens()["total"]
tokes_per_itr = config.batch_size * config.context_len * config.grad_acum_steps * dpp_info.world_size
print(f"Will train using on {device}:\
     \n DPP Info:\n  use dpp: {dpp_info.ddp}\n  world_size: {dpp_info.world_size}\n  rank: {dpp_info.rank}\n  local_rank: {dpp_info.local_rank}\n  master_process: {dpp_info.master_process}\
     \nRuns {MAX_ITRS} training steps and each step runs for {config.grad_acum_steps} iterations processing {tokes_per_itr} tokens")

print("Token distribution  "+str(data_loader.total_tokens()))
print(f"Total training tokens {data_loader.total_tokens()["total"]}, requires min {data_loader.total_tokens()["total"] // tokes_per_itr} steps or {data_loader.total_tokens()["total"] // (config.batch_size * config.context_len )} iterations (microsteps)")

Will train using on mps:     
 DPP Info:
  use dpp: False
  world_size: 1
  rank: 0
  local_rank: 0
  master_process: True     
Runs 20 training steps and each step runs for 64 iterations processing 524288 tokens
Token distribution  {'fineweb': 90000000, 'books': 90000000, 'tinystories': 90000000, 'total': 270000000}
Total training tokens 270000000, requires min 514 steps or 32958 iterations (microsteps)


In [19]:
trainer.load_checkpoint(path="/Users/ashritkuma.samudrala/lnex/ex_llm_rag/main/resources/models/gpt2/gp2_model_check_point_train.model")

Resumed from /Users/ashritkuma.samudrala/lnex/ex_llm_rag/main/resources/models/gpt2/gp2_model_check_point_train.model — starting at step 2


In [36]:
trainer.train_model(10, use_ddp=dpp_info.ddp, master_process=dpp_info.master_process, world_size=dpp_info.world_size, time_itr=True) # 30.4s

Step 0,  loss = 7.2102 | dt = 159173.15 ms | norm = 0.4179 | LR = 6.00e-05 | token_throughput = 3293.82 tokens/s


In [43]:
# Checkpoint 
ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
path = f"{ROOT}/main/resources/models/gpt2/gpt2_20steps.model"
os.makedirs(os.path.dirname(path), exist_ok=True)

# out = trainer.estimate_loss()
torch.save({
    'iter'                : config.max_train_iters,
    'model_state_dict'    : {k: v.cpu() for k, v in model.state_dict().items()},
    'optimizer_state_dict': trainer.optim.state_dict(),
    'gpt_config'          : config,
    'val_loss'            : 7.2,
}, path)

print(path)

/Users/ashritkuma.samudrala/lnex/ex_llm_rag/main/resources/models/gpt2/gpt2_20steps.model


In [20]:
# Sample from the trained model
GPT2.sample(model, start_text="Hello I'm a laguage model,", max_len=50, n_samples=1, seed=44)

Hello I'm a laguage model, Fees filtered underwent randomizeddies swing VitalckiFuel anal 2010 ThornTokens effortadd132 thous LB Xia 415 attRot RightsYoung November------------------------------------------------ commissioner organizingBio signific Teachers Thief resideDoc residues via overflowing014 selected launcher296 burnt


["Hello I'm a laguage model, Fees filtered underwent randomizeddies swing VitalckiFuel anal 2010 ThornTokens effortadd132 thous LB Xia 415 attRot RightsYoung November------------------------------------------------ commissioner\x1f organizingBio signific Teachers Thief resideDoc residues via overflowing014 selected launcher296 burnt"]